## Working with timeseries

This notebook runs our ["Working with Timeseries"](https://docs.lsdb.io/en/latest/tutorials/pre_executed/timeseries.html) workflow with both Dask and Ray.

In [1]:
import lsdb
import numpy as np
from nested_pandas.utils import count_nested
from astropy.timeseries import LombScargle

We will compute the periodogram for ZTF DR22 objects with:

- perfectly clean extractions (lc.catflags == 0)
- r-band only (filterid == 2)
- median of lc.mag is brighter than 16
- at least 10 points

In [2]:
ztf_objects = lsdb.open_catalog(
    "/mnt/data/hats/catalogs/ztf_dr22",
).nest_lists(
    list_columns=["hmjd", "mag", "magerr", "clrcoeff", "catflags"],
    name="lc",  # name to give the resulting nested column
)

filtered_catalog = ztf_objects.query("filterid == 2")
filtered_catalog = filtered_catalog.query("lc.catflags == 0")
# Using map_partitions to drop rows with NaN in lc.mag
filtered_catalog = filtered_catalog.map_partitions(lambda df: df.dropna(subset=["lc.mag"]))

def median_mag(row):
    return np.median(row["lc.mag"])

catalog_w_features = filtered_catalog.map_rows(
    median_mag,  # our user-defined function
    columns=["lc.mag"],  # names of the column(s) to pass to the function
    output_names=["median_mag"],  # name(s) of the new column(s)
    meta={"median_mag": float},  # for Dask: describe the type of the output
    append_columns=True,
)

catalog_w_features = catalog_w_features.query("median_mag < 16")
catalog_w_features = catalog_w_features.map_partitions(lambda df: count_nested(df, "lc"))
catalog_w_features = catalog_w_features.query("n_lc >= 10")

def extract_period(row):
    time = row["lc.hmjd"]
    mag = row["lc.mag"]
    error = row["lc.magerr"]
    ls = LombScargle(time, mag, error)
    freq, power = ls.autopower()
    argmax = np.argmax(power)
    period = 1.0 / freq[argmax]
    false_alarm_prob = ls.false_alarm_probability(power[argmax])
    return {"period": period, "false_alarm_prob": false_alarm_prob}

catalog_w_features = catalog_w_features.map_rows(
    extract_period,
    # Column names specifying function arguments
    columns=["lc.hmjd", "lc.mag", "lc.magerr"],
    # Returned data type shape
    meta={"period": float, "false_alarm_prob": float},
    append_columns=True,
)
periodic_catalog = catalog_w_features.query("false_alarm_prob < 1e-10 and 1.5 < period < 9.5")

periodic_catalog

,objectid,filterid,objra,objdec,nepochs,lc,median_mag,n_lc,period,false_alarm_prob
npartitions=10839,,,,,,,,,,
"Order: 4, Pixel: 0",int64[pyarrow],int8[pyarrow],float[pyarrow],float[pyarrow],int64[pyarrow],"nested<hmjd: [double], mag: [float], magerr: [...",float64,int32[pyarrow],float64,float64
"Order: 4, Pixel: 1",...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 12286",...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 12287",...,...,...,...,...,...,...,...,...,...


This workflow includes calls to several core operations:
- `open_catalog`
- `map_partitions`
- `map_rows`
- `query`

### Compute with Dask

In [3]:
from dask.distributed import Client

In [4]:
%%time
with Client(n_workers=2):
    periodic_catalog.partitions[:2].compute()

/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


CPU times: user 20.5 s, sys: 2.15 s, total: 22.6 s
Wall time: 2min 43s


<img src="assets/dask-timeseries.png" width=1000>

### Compute with Ray


In [5]:
import ray
from ray.util.dask import enable_dask_on_ray

In [6]:
%%time
ray.init(num_cpus=2)
with enable_dask_on_ray():
    periodic_catalog.partitions[:2].compute()
ray.shutdown()

2026-04-10 13:12:37,016	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/dask/config.py:835: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


Computing Catalog:   0%|          | 0/31 [00:00<?, ?it/s]

(dask:('lambda-418c51914d45b4a8f2acb18c7de8a4b3', 1) pid=3211246) /home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
(dask:('lambda-418c51914d45b4a8f2acb18c7de8a4b3', 1) pid=3211246)   return _methods._mean(a, axis=axis, dtype=dtype,
(dask:('lambda-418c51914d45b4a8f2acb18c7de8a4b3', 1) pid=3211246) /home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
(dask:('lambda-418c51914d45b4a8f2acb18c7de8a4b3', 1) pid=3211246)   ret = ret.dtype.type(ret / rcount)
(dask:('lambda-418c51914d45b4a8f2acb18c7de8a4b3', 0) pid=3211245) /home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
(dask:('lambda-418c51914d45b4a8f2acb18c7de8a4b3', 0) p

CPU times: user 8.58 s, sys: 599 ms, total: 9.18 s
Wall time: 2min 53s


<img src="assets/ray-mem.png" width=800>

<img src="assets/ray-store.png" width=800>